# NLLB-200-600M LoRA Finetuning — Arabic ↔ Hindi

Finetune `facebook/nllb-200-distilled-600M` on 20k Ar–Hi pairs with **LoRA**, then evaluate 500 held-out pairs with **BLEU**, **chrF++**, and **COMET**.

- **Direction:** Arabic (`arb_Arab`) → Hindi (`hin_Deva`). Flip `SRC_LANG`/`TGT_LANG` in `CFG` to reverse.
- **Runtime:** ~30–45 min on a T4 (3 epochs).
- **Requirements:** GPU enabled, Internet ON, ~2 GB free in `/kaggle/working` for the COMET model.

## 0. Install

In [1]:
!pip install -q -U transformers datasets accelerate peft sentencepiece
!pip install -q sacrebleu unbabel-comet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 1.17.0 which is incompatible.
unbabel-comet 2.2.7 requires transformers<5.0,>=4.17, but you have transformers 5.9.0 which is incompatible.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


## 1. Config

Single source of truth for paths, languages, and hyperparams. Tweak here, leave the rest alone.

In [2]:
import os, gc, random, torch, numpy as np, pandas as pd
from dataclasses import dataclass

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

@dataclass
class CFG:
    # --- paths (EDIT THESE) ---
    DATA_PATH:   str = "/kaggle/input/datasets/mishbhaul/train-and-test/train_parallel_corpus.csv"
    OUTPUT_DIR:  str = "/kaggle/working/nllb-lora-run"
    ADAPTER_DIR: str = "/kaggle/working/nllb-lora-adapter"

    # --- columns in your CSV ---
    SRC_COL: str = "hi"
    TGT_COL: str = "ar"

    # --- model + NLLB language codes ---
    MODEL_NAME: str = "facebook/nllb-200-distilled-600M"
    SRC_LANG:   str = "arb_Arab"
    TGT_LANG:   str = "hin_Deva"

    # --- split sizes ---
    N_TRAIN: int = 20100
    N_EVAL:  int = 0

    # --- training ---
    SEED:         int   = 42
    MAX_LENGTH:   int   = 128
    BATCH_SIZE:   int   = 8
    GRAD_ACCUM:   int   = 1
    LR:           float = 2e-4   # LoRA tolerates higher LR than full FT
    EPOCHS:       int   = 3
    WARMUP_RATIO: float = 0.1
    WEIGHT_DECAY: float = 0.01

    # --- LoRA ---
    LORA_R:       int   = 32
    LORA_ALPHA:   int   = 32
    LORA_DROPOUT: float = 0.05
    LORA_TARGETS: tuple = ("q_proj", "k_proj", "v_proj", "out_proj")

    # --- inference ---
    NUM_BEAMS: int = 5

cfg = CFG()
set_seed(cfg.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## 2. Load & Split Data

Reads your CSV, deduplicates, shuffles, and splits 20k train / 500 eval.

In [3]:
df = pd.read_csv(cfg.DATA_PATH)
df = df[[cfg.SRC_COL, cfg.TGT_COL]].dropna().drop_duplicates().reset_index(drop=True)
print(f"Total clean pairs: {len(df):,}")
df.head(3)

Total clean pairs: 20,100


,hi,ar
0,आगामी व्यापार समझौते से इंडोनेशिया में प्रत्यक...,وستفتح هذه الاتفاقية التجارية آفاقا واسعة للاس...
1,"लेबनान में, सरकार ने चल रहे इज़राइल-लेबनान संघ...",في لبنان، أبرمت الحكومة اتفاقًا مع ستارلينك لض...
2,"भले ही कोई पाठक लेखन की एक शैली से परिचित हो, ...",حتى لو كان القارئ معتادًا على أسلوب كتابة معين...


In [4]:
from datasets import Dataset

df = df.sample(frac=1, random_state=cfg.SEED).reset_index(drop=True)
assert len(df) >= cfg.N_TRAIN + cfg.N_EVAL, "Not enough rows for the requested split"

train_df = df.iloc[:cfg.N_TRAIN].reset_index(drop=True)
eval_df  = df.iloc[cfg.N_TRAIN : cfg.N_TRAIN + cfg.N_EVAL].reset_index(drop=True)

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
eval_ds  = Dataset.from_pandas(eval_df,  preserve_index=False)
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

Train: 20100 | Eval: 0


## 3. Load Model & Tokenizer

NLLB needs the source and target language set on the tokenizer.

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(
    cfg.MODEL_NAME, src_lang=cfg.SRC_LANG, tgt_lang=cfg.TGT_LANG
)
model = AutoModelForSeq2SeqLM.from_pretrained(cfg.MODEL_NAME)
print(f"Loaded {cfg.MODEL_NAME}  ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

2026-05-29 17:07:43.050386: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780074463.297917     148 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780074463.369084     148 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780074463.962014     148 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780074463.962040     148 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780074463.962043     148 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded facebook/nllb-200-distilled-600M  (615.1M params)


## 4. Apply LoRA

Adapts only the attention projections — trainable params drop to <1%.

In [6]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.3 MB/s eta 0:00:00a 0:00:01


In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=cfg.LORA_R,
    lora_alpha=cfg.LORA_ALPHA,
    lora_dropout=cfg.LORA_DROPOUT,
    bias="none",
    target_modules=list(cfg.LORA_TARGETS),
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainable params: 9,437,184 || all params: 624,510,976 || trainable%: 1.5111


## 5. Tokenize

In [8]:
def preprocess(batch):
    return tokenizer(
        batch[cfg.SRC_COL],
        text_target=batch[cfg.TGT_COL],
        max_length=cfg.MAX_LENGTH,
        truncation=True,
    )

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map( preprocess, batched=True, remove_columns=eval_ds.column_names)
print(train_tok)

Map:   0%|          | 0/20100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 20100
})


## 6. Train

In [9]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

args = Seq2SeqTrainingArguments(
    output_dir=cfg.OUTPUT_DIR,
    num_train_epochs=cfg.EPOCHS,
    per_device_train_batch_size=cfg.BATCH_SIZE,
    per_device_eval_batch_size=cfg.BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRAD_ACCUM,
    learning_rate=cfg.LR,
    weight_decay=cfg.WEIGHT_DECAY,
    warmup_ratio=cfg.WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    eval_strategy="no",          # final eval is done separately (so we can run COMET too)
    report_to="none",
    predict_with_generate=False,
    remove_unused_columns=False,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding="longest")

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    processing_class=tokenizer,
    data_collator=collator,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,3.250600
100,2.991900
150,2.380100
200,2.075600
250,1.818200
300,1.754500
350,1.746400
400,1.737200
450,1.760100
500,1.714700


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=3771, training_loss=1.7218075703954987, metrics={'train_runtime': 5501.3972, 'train_samples_per_second': 10.961, 'train_steps_per_second': 0.685, 'total_flos': 1.474024303460352e+16, 'train_loss': 1.7218075703954987, 'epoch': 3.0})

## 7. Save the LoRA Adapter

Only the adapter is saved (~10–30 MB) — the base model stays on the Hub.

In [10]:
model.save_pretrained(cfg.ADAPTER_DIR)
tokenizer.save_pretrained(cfg.ADAPTER_DIR)
print("Saved →", cfg.ADAPTER_DIR)

Saved → /kaggle/working/nllb-lora-adapter


## 8. Inference

NLLB needs `forced_bos_token_id` at generate time, or it outputs the wrong language.

In [11]:
from typing import List

@torch.inference_mode()
def translate(texts: List[str], batch_size: int = 16) -> List[str]:
    model.eval()
    tokenizer.src_lang = cfg.SRC_LANG
    fb_id = tokenizer.convert_tokens_to_ids(cfg.TGT_LANG)
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        enc = tokenizer(chunk, return_tensors="pt", padding=True,
                        truncation=True, max_length=cfg.MAX_LENGTH).to(model.device)
        gen = model.generate(
            **enc,
            forced_bos_token_id=fb_id,
            max_length=cfg.MAX_LENGTH,
            num_beams=cfg.NUM_BEAMS,
        )
        out.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return out

# sanity check on 3 samples
for s, r, h in zip(eval_df[cfg.SRC_COL].iloc[:3],
                   eval_df[cfg.TGT_COL].iloc[:3],
                   translate(eval_df[cfg.SRC_COL].iloc[:3].tolist())):
    print(f"SRC: {s}\nREF: {r}\nHYP: {h}\n{'-'*60}")

## 9. Generate Predictions on the 500-pair Eval Set

In [14]:
eval_df  = pd.read_csv("/kaggle/input/datasets/mishbhaul/train-and-test/devtest_parallel_corpus.csv")

In [15]:
sources = eval_df[cfg.SRC_COL].tolist()
refs    = eval_df[cfg.TGT_COL].tolist()

print(f"Translating {len(sources)} eval samples...")
hyps = translate(sources, batch_size=16)

eval_out = eval_df.copy()
eval_out["prediction"] = hyps
eval_out.to_csv("/kaggle/working/eval_predictions.csv", index=False)
eval_out.head()

Translating 500 eval samples...


,ar,hi,prediction
0,جاء هذا بينما أوضح مسؤول في حماس، أن دراسة خطة...,"समाचार एजेंसी रॉयटर्स की रिपोर्ट के अनुसार, हम...",وقال مسؤول في حركة حماس إنه قد يستغرق دراسة خط...
1,بدوره، وصف الرئيس الإيراني، مسعود بزشكيان الأس...,संयुक्त राज्य अमेरिका की हालिया मांगों के जवाब...,وقال الرئيس الإيراني مسعود بيزيشكيان، رداً على...
2,"وأعلنت الوزارة عن ارتفاع ""مدروس"" لمحفظة الدين ...",मंत्रालय ने वित्तीय वर्ष 2026 ई. के समापन पर द...,كما أعلنت الوزارة عن التخطيط لزيادة محفظة الدي...
3,وكان سان جيرمان الذي توج بلقب أبطال أوروبا في ...,महाद्वीपीय प्रतियोगिता के आगामी 2024 संस्करण म...,وفي الدورة القادمة من المسابقة القارية لعام 20...
4,وفي سياق متصل، استعرض المجلس تطور خدمات قطاع ا...,"संबंधित संदर्भ में, परिषद ने राज्य के सभी क्षे...",وفي هذا السياق، فحص المجلس تقدم الخدمات في قطا...


## 10. BLEU & chrF++ (sacrebleu)

`word_order=2` in `corpus_chrf` is what makes it chrF++ (default is chrF).

In [16]:
import sacrebleu

bleu   = sacrebleu.corpus_bleu(hyps, [refs])
chrfpp = sacrebleu.corpus_chrf(hyps, [refs], word_order=2)

print(f"BLEU   : {bleu.score:.2f}")
print(f"chrF++ : {chrfpp.score:.2f}")

BLEU   : 13.33
chrF++ : 43.15


## 11. COMET (`Unbabel/wmt22-comet-da`)

Reference-based neural metric — usually correlates better with humans than BLEU. First run downloads ~2 GB; cached afterwards.

In [17]:
from comet import download_model, load_from_checkpoint

# free up VRAM before loading COMET on the same GPU
del trainer; gc.collect(); torch.cuda.empty_cache()

comet_path  = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

comet_data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(sources, hyps, refs)]
comet_out  = comet_model.predict(
    comet_data, batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0,
)
print(f"COMET  : {comet_out.system_score:.4f}")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packag

COMET  : 0.8351


## 12. Summary

In [18]:
results = pd.DataFrame([{
    "model":       cfg.MODEL_NAME,
    "direction":   f"{cfg.SRC_LANG} → {cfg.TGT_LANG}",
    "train_pairs": cfg.N_TRAIN,
    "eval_pairs":  cfg.N_EVAL,
    "BLEU":        round(bleu.score, 2),
    "chrF++":      round(chrfpp.score, 2),
    "COMET":       round(comet_out.system_score, 4),
}])
results.to_csv("/kaggle/working/results_summary.csv", index=False)
results

,model,direction,train_pairs,eval_pairs,BLEU,chrF++,COMET
0,facebook/nllb-200-distilled-600M,arb_Arab → hin_Deva,20100,0,13.33,43.15,0.8351


---

### Reloading the adapter later

```python
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

base = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
tok  = AutoTokenizer.from_pretrained("/kaggle/working/nllb-lora-adapter",
                                     src_lang="arb_Arab", tgt_lang="hin_Deva")
model = PeftModel.from_pretrained(base, "/kaggle/working/nllb-lora-adapter")
```